In [2]:
# !pip install sentence-transformers pinecone-client google-generativeai beautifulsoup4

from pathlib import Path
import time, os, re, json
from typing import List, Dict, Tuple
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from google import generativeai as genai

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "memoryindex"
REGION = "us-east-1"
DIM = 384

In [4]:
existing = [d["name"] for d in pc.list_indexes()]
if INDEX_NAME in existing:
    try:
        pc.delete_index(name=INDEX_NAME)
        while INDEX_NAME in [d["name"] for d in pc.list_indexes()]:
            time.sleep(1)
    except Exception as e:
        print("Warning: failed to delete existing index:", e)

pc.create_index(
    name=INDEX_NAME,
    dimension=DIM,
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region=REGION)
)

# Get a handle to the index
index = pc.Index(INDEX_NAME)

# Namespaces: docs for content; mem_user for long-term memory
DOCS_NS = "docs"
USER_ID = "user_001"
MEM_NS  = f"mem_{USER_ID}"

# Embedding model (384-dim)
EMBED_MODEL = "paraphrase-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

# Gemini LLM
genai.configure(api_key=GOOGLE_API_KEY)
llm = genai.GenerativeModel("gemini-2.5-flash")

In [5]:
import yaml

def load_prompt_template(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return data["template"]

PROMPT_TEMPLATE = load_prompt_template("prompts/memory_rag.yaml")

In [6]:
txt_dir = Path("coffee_txt")

def extract_heading_keywords(headings: List[str]) -> List[str]:
    # basic tokenization for demo; in prod, normalize, drop stopwords, etc.
    toks = []
    for h in headings:
        toks += re.findall(r"[A-Za-z][A-Za-z\-]+", h)
    return sorted({t.lower() for t in toks})

def upsert_docs_from_txt(directory: Path) -> int:
    pairs = []
    for fp in directory.glob("*.txt"):
        raw = fp.read_text(encoding="utf-8")

        # derive a title from the first non-empty line, fallback to filename
        first_line = ""
        for line in raw.splitlines():
            if line.strip():
                first_line = line.strip()
                break
        title = first_line or fp.stem

        # full text as document content
        page_text = raw

        # embedding
        vec = embedder.encode([page_text], normalize_embeddings=True)[0].tolist()

        # metadata
        meta = {
            "file_name": fp.name,
            "title": title,
            "headings": "",
            "heading_keywords": extract_heading_keywords([title]),
            "ts": int(time.time()),
            "page_text": page_text
        }

        doc_id = fp.stem
        pairs.append((doc_id, vec, meta))

    if pairs:
        index.upsert(vectors=pairs, namespace=DOCS_NS)
    return len(pairs)

indexed = upsert_docs_from_txt(txt_dir)
print(f"Indexed {indexed} text files into namespace '{DOCS_NS}'.")

Indexed 12 text files into namespace 'docs'.


In [7]:
def extract_user_facts(user_text: str) -> List[str]:
    """
    Demo-only extractor. In production:
      - use an LLM-based information extractor,
      - define a strict JSON schema for {preference, value, evidence, ts}.
    """
    pats = [
        r"\bI (?:like|love|prefer)\b[^.]+",
        r"\bI (?:avoid|usually|often|am)\b[^.]+",
        r"\bMy [A-Za-z ]+\b[^.]+"
    ]
    findings = []
    for pat in pats:
        findings += [m.group(0).strip() for m in re.finditer(pat, user_text, flags=re.I)]
    return sorted(set(findings))

def add_memory_facts(facts: List[str]) -> None:
    if not facts:
        return
    vecs = embedder.encode(facts, normalize_embeddings=True).tolist()
    now = int(time.time())
    payload = []
    for i, fact in enumerate(facts):
        vid = f"{USER_ID}:{now}:{i}"
        meta = {"fact": fact, "user_id": USER_ID, "ts": now}
        payload.append((vid, vecs[i], meta))
    index.upsert(vectors=payload, namespace=MEM_NS)

def retrieve_memory(query_text: str, k: int = 3) -> List[Dict]:
    qv = embedder.encode([query_text], normalize_embeddings=True)[0].tolist()
    res = index.query(
        vector=qv, top_k=k, include_values=False, include_metadata=True, namespace=MEM_NS
    )
    return res.get("matches", [])

In [8]:
def retrieve_docs(query_text: str, k: int = 5, keywords: List[str] = None) -> List[Dict]:
    qv = embedder.encode([query_text], normalize_embeddings=True)[0].tolist()
    filt = {"heading_keywords": {"$in": [kw.lower() for kw in keywords]}} if keywords else None
    res = index.query(
        vector=qv, top_k=k, include_values=False, include_metadata=True,
        namespace=DOCS_NS, filter=filt
    )
    return res.get("matches", [])

In [9]:
def format_history(history: List[Dict]) -> str:
    if not history: return "None"
    return "\n".join(
        f"{'User' if t['role']=='user' else 'Assistant'}: {t['content']}"
        for t in history
    )

def format_memory(mem_hits: List[Dict]) -> str:
    if not mem_hits: return "None"
    return "\n".join(f"- {h['metadata'].get('fact','')}" for h in mem_hits)

def format_context(doc_hits: List[Dict]) -> str:
    if not doc_hits: return "None"
    lines = []
    for h in doc_hits:
        doc_id = h.get("id", "")
        meta = h.get("metadata", {})
        title = meta.get("title") or meta.get("file_name") or "Untitled"
        page_text = meta.get("page_text", "")
        lines.append(f"[{doc_id}] {title} - {page_text}")
    return "\n".join(lines)

def build_prompt(query: str, history: List[Dict], doc_hits: List[Dict], mem_hits: List[Dict]) -> str:
    return PROMPT_TEMPLATE.format(
        history=format_history(history),
        memory_block=format_memory(mem_hits),
        query=query,
        context_block=format_context(doc_hits)
    )

In [10]:
history: List[Dict] = []

def rag_turn(user_text: str, k_docs: int = 5, k_mem: int = 3, keywords: List[str] = None) -> str:
    # 1) long-term memory update from user message
    new_facts = extract_user_facts(user_text)
    add_memory_facts(new_facts)

    # 2) retrieve current docs + memory
    doc_hits = retrieve_docs(user_text, k=k_docs, keywords=keywords)
    mem_hits = retrieve_memory(user_text, k=k_mem)
    # 3) assemble the prompt from template
    prompt = build_prompt(user_text, history, doc_hits, mem_hits)

    # 4) call the LLM
    resp = llm.generate_content(prompt)
    answer = getattr(resp, "text", "").strip()

    # 5) update short-term memory (conversation history)
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": answer})

    # 6) tracing for teaching
    print("---- DOCS ----")
    for h in doc_hits:
        print(h["id"], round(h["score"],3), "--", h["metadata"].get("title") or h["metadata"].get("file_name"))
    print("---- MEMORY ----")
    for h in mem_hits:
        print("-", h["metadata"]["fact"])
    print("---- ANSWER ----")
    print(answer, "\n")

    return answer    

In [11]:
print("Memory-RAG. Type 'exit' to quit.\n")

while True:
    q = input("You: ").strip()
    print("Query:", q)
    if q.lower() in {"exit","quit"}:
        print("Goodbye!")
        break
    _ = rag_turn(q)

Memory-RAG. Type 'exit' to quit.

Query: hello
---- DOCS ----
cold-brew 0.15 -- Cold Brew
flat-white 0.146 -- Flat White
nitro-cold-brew 0.134 -- Nitro Cold Brew
french-press 0.105 -- French Press
mocha 0.104 -- Mocha (duplicate sample)
---- MEMORY ----
---- ANSWER ----
Hello there! As your friendly coffee expert, I'm ready to help you with any coffee questions you have. What can I brew up for you today? 

Query: I like coffee with very low caffine. Can you suggest some coffee?
---- DOCS ----
flat-white 0.492 -- Flat White
cold-brew 0.423 -- Cold Brew
french-press 0.414 -- French Press
latte 0.4 -- Caffè Latte
cappuccino 0.389 -- Cappuccino
---- MEMORY ----
- I like coffee with very low caffine
---- ANSWER ----
For coffee with very low caffeine, a **Caffè Latte** or a **Cappuccino** could be good options, as they can be made with a single shot of espresso, providing around 63 mg of caffeine [latte] [cappuccino].

Here's why:
*   **Caffè Latte:** Emphasizes milk texture with a single sh